# Model 3: DistilBERT on Google Colab

Select **Runtime → Change runtime type → T4 GPU**, then run the cells in order.
Upload `train.parquet` and `validation.parquet` from this project when prompted.
The model uses only review descriptions and predicts red, white, sparkling, rose,
or dessert. The saved test split remains reserved for final evaluation.

This experiment uses training-only inverse-frequency class weights, dynamic
padding, mixed precision, and up to three epochs. The selected checkpoint has
the highest validation macro F1. There is no guarantee it will beat the SVM.

References: [Hugging Face text classification](https://huggingface.co/docs/transformers/en/tasks/sequence_classification)
and [Trainer customization](https://huggingface.co/docs/transformers/v4.57.1/en/main_classes/trainer).

Run the download cell before ending the Colab session. A new classification head
is initialized randomly; its initialization warning is expected.


**1. Install packages** — run this before importing Transformers in a fresh runtime.

In [ ]:
%pip -q install "transformers==4.57.6" "datasets>=3,<5" "accelerate>=1,<2" scikit-learn pandas pyarrow

**2. Check the GPU and upload the two saved splits.**

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"

import json
import shutil
import time
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from google.colab import files
from datasets import Dataset
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    classification_report, confusion_matrix,
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, Trainer, TrainingArguments,
    EarlyStoppingCallback, set_seed,
)

assert torch.cuda.is_available(), "Select Runtime > Change runtime type > T4 GPU."
print("GPU:", torch.cuda.get_device_name(0))
set_seed(42)

if not all(Path(name).exists() for name in ["train.parquet", "validation.parquet"]):
    print("Select train.parquet and validation.parquet together.")
    files.upload()
assert all(Path(name).exists() for name in ["train.parquet", "validation.parquet"])

**3. Validate the data and tokenize descriptions.** Labels and class weights are derived from training data only.

In [ ]:
COLORS = ["red", "white", "sparkling", "rose", "dessert"]
label2id = {color: i for i, color in enumerate(COLORS)}
id2label = {i: color for color, i in label2id.items()}
MODEL_NAME = "distilbert/distilbert-base-uncased"
MAX_LENGTH = 128

train_df = pd.read_parquet("train.parquet")
val_df = pd.read_parquet("validation.parquet")
train_df = train_df.loc[train_df["color"].isin(COLORS)].copy()
val_df = val_df.loc[val_df["color"].isin(COLORS)].copy()

for frame in [train_df, val_df]:
    assert not frame.empty
    assert set(frame["color"]) == set(COLORS), "A wine category is missing."
    assert frame["description"].notna().all(), "Missing descriptions."
    assert frame["description"].map(lambda text: isinstance(text, str)).all()
    assert frame["description"].str.strip().ne("").all(), "Empty descriptions."
    frame["labels"] = frame["color"].map(label2id).astype("int64")

normalize = lambda series: series.str.strip().str.casefold()
assert set(normalize(train_df["description"])).isdisjoint(
    set(normalize(val_df["description"]))
), "Training and validation contain overlapping descriptions."

counts = train_df["labels"].value_counts().reindex(range(len(COLORS)))
class_weights = torch.tensor(
    len(train_df) / (len(COLORS) * counts.to_numpy()), dtype=torch.float32
)
print(pd.DataFrame({
    "train": train_df["color"].value_counts(),
    "validation": val_df["color"].value_counts(),
}).reindex(COLORS))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
def tokenize(batch):
    return tokenizer(batch["description"], truncation=True, max_length=MAX_LENGTH)

def make_dataset(frame):
    dataset = Dataset.from_pandas(frame[["description", "labels"]], preserve_index=False)
    return dataset.map(tokenize, batched=True, remove_columns=["description"])

train_ds = make_dataset(train_df)
val_ds = make_dataset(val_df)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

**4. Configure the model and class-weighted training.**

In [ ]:
class WeightedTrainer(Trainer):
    def __init__(self, *args, class_weights, **kwargs):
        super().__init__(*args, **kwargs)
        self.class_weights = class_weights
        self.model_accepts_loss_kwargs = False

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(
            outputs.logits.float(), labels,
            weight=self.class_weights.to(outputs.logits.device),
        )
        return (loss, outputs) if return_outputs else loss

def compute_metrics(prediction):
    predicted = prediction.predictions.argmax(axis=-1)
    actual = prediction.label_ids
    return {
        "accuracy": accuracy_score(actual, predicted),
        "balanced_accuracy": balanced_accuracy_score(actual, predicted),
        "macro_f1": f1_score(
            actual, predicted, labels=list(id2label), average="macro", zero_division=0
        ),
    }

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(COLORS), id2label=id2label, label2id=label2id,
)
use_bf16 = torch.cuda.get_device_capability(0)[0] >= 8 and torch.cuda.is_bf16_supported()
args = TrainingArguments(
    output_dir="wine_distilbert_checkpoints",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=1,
    num_train_epochs=3,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    save_total_limit=2,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=100,
    report_to="none",
    push_to_hub=False,
    seed=42,
)
trainer = WeightedTrainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    processing_class=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    class_weights=class_weights,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

**5. Train and evaluate the selected checkpoint.** The progress bar estimates remaining time after training starts.

In [ ]:
started = time.perf_counter()
train_result = trainer.train()
elapsed_minutes = (time.perf_counter() - started) / 60
print(f"Training plus epoch validation: {elapsed_minutes:.1f} minutes")
print("Selected checkpoint:", trainer.state.best_model_checkpoint)

result = trainer.predict(val_ds, metric_key_prefix="validation")
actual = result.label_ids
predicted = result.predictions.argmax(axis=-1)
print(classification_report(
    actual, predicted, labels=list(id2label), target_names=COLORS,
    digits=3, zero_division=0,
))
confusion = pd.DataFrame(
    confusion_matrix(actual, predicted, labels=list(id2label)),
    index=[f"actual {color}" for color in COLORS],
    columns=[f"predicted {color}" for color in COLORS],
)
display(confusion)
display(pd.DataFrame([
    {"model": "Original SVM", "validation_macro_f1": 0.7370675324},
    {"model": "Weighted SVM (C=0.1)", "validation_macro_f1": 0.7745722448},
    {"model": "DistilBERT", "validation_macro_f1": result.metrics["validation_macro_f1"]},
]))

**6. Save and download the selected model, tokenizer, and results.** Intermediate checkpoints are excluded from this ZIP.

In [ ]:
export_dir = Path("wine_distilbert_export")
export_dir.mkdir(exist_ok=True)
trainer.save_model(str(export_dir / "model"))
tokenizer.save_pretrained(str(export_dir / "model"))

predictions = pd.DataFrame({
    "row_id": val_df.index.to_numpy(),
    "actual_color": val_df["color"].to_numpy(),
    "predicted_color": [id2label[int(i)] for i in predicted],
    "description": val_df["description"].to_numpy(),
})
predictions.to_csv(export_dir / "validation_predictions.csv", index=False)
confusion.to_csv(export_dir / "confusion_matrix.csv")
(export_dir / "metrics.json").write_text(json.dumps(result.metrics, indent=2))
(export_dir / "training_history.json").write_text(json.dumps(trainer.state.log_history, indent=2))
(export_dir / "training_settings.json").write_text(json.dumps({
    "model": MODEL_NAME, "max_length": MAX_LENGTH, "seed": 42,
    "training_rows": len(train_df), "validation_rows": len(val_df),
    "label2id": label2id, "class_weights": class_weights.tolist(),
    "training_arguments": args.to_dict(),
    "elapsed_minutes": elapsed_minutes,
    "best_checkpoint": trainer.state.best_model_checkpoint,
}, indent=2))

archive = shutil.make_archive("wine_distilbert_results", "zip", root_dir=export_dir)
files.download(archive)